# Rasch Mixed Item Cut Score Computation (PCM)

This notebook provides a complete walk-through to compute the person ability measure ($\theta$ in logits) corresponding to a given raw cut score on a test containing a mixture of **dichotomous** and **polytomous** items under the **Rasch model**.

Polytomous items are modeled using the **Partial Credit Model (PCM)** (Masters, 1982). Dichotomous items are modeled as a special case of PCM with a single step difficulty ($m_i = 1$).

---

## Mathematical Formulation

### 1. The Partial Credit Model (PCM)

For an item $i$ with category scores $x \in \{0, 1, \dots, m_i\}$ and absolute step difficulties $\boldsymbol{\delta}_i = [\delta_{i1}, \delta_{i2}, \dots, \delta_{im_i}]$, the probability of a person with ability $\theta$ scoring in category $x$ is defined by the adjacent-category log-odds:

$$\ln \left( \frac{P_{ix}(\theta)}{P_{i(x-1)}(\theta)} \right) = \theta - \delta_{ix}$$

Solving this recurrence relation with the constraint that the probabilities sum to $1$ yields the category response probability function:

$$P_{ix}(\theta) = \frac{\exp \left( \sum_{j=1}^x (\theta - \delta_{ij}) \right)}{\sum_{k=0}^{m_i} \exp \left( \sum_{j=1}^k (\theta - \delta_{ij}) \right)}$$

where the summation in the exponent is defined to be $0$ when the upper limit is $0$ ($x=0$ or $k=0$):
$$\sum_{j=1}^0 (\theta - \delta_{ij}) = 0 \implies e^0 = 1$$

### 2. Test Characteristic Curve (TCC) & Test Information Function (TIF)

The expected score $E_i(\theta)$ of item $i$ at ability $\theta$ is:
$$E_i(\theta) = \sum_{x=0}^{m_i} x \cdot P_{ix}(\theta)$$

The Test Characteristic Curve (TCC) represents the expected raw test score as a function of person ability $\theta$:
$$TCC(\theta) = \sum_{i=1}^N E_i(\theta)$$

The first derivative of the TCC with respect to $\theta$ gives the Test Information Function (TIF), which represents the slope of the TCC and is the sum of the individual item response variances:
$$I(\theta) = TCC'(\theta) = \sum_{i=1}^N Var_i(\theta) = \sum_{i=1}^N \left[ \sum_{x=0}^{m_i} x^2 P_{ix}(\theta) - (E_i(\theta))^2 \right]$$

The conditional standard error of measurement (CSEM) at ability $\theta$ is the reciprocal square root of the information:
$$SE(\theta) = \frac{1}{\sqrt{I(\theta)}}$$

---

## Solving for the Logit Cut Score

Under Rasch measurement, person ability $\theta$ corresponding to a raw score $X$ is found by solving the following equation:
$$TCC(\theta) = X$$

Since $TCC(\theta)$ is strictly increasing ($TCC'(\theta) > 0$), there is a unique real solution for any raw score in the open interval $(0, M)$ (where $M = \sum_i m_i$ is the maximum possible score).

### 1. Extreme Score Adjustment (Winsteps Style)
For extreme raw scores (zero or perfect scores), a direct maximum likelihood estimate (MLE) is infinite. To provide finite, useful measures, Winsteps applies an extreme score adjustment (`extrscore = 0.3` by default):
$$X_{\text{adjusted}} = \begin{cases} \text{extrscore} & \text{if } X \le 0 \\ M - \text{extrscore} & \text{if } X \ge M \\ X & \text{otherwise} \end{cases}$$

### 2. Newton-Raphson Optimization
Starting from an initial ability estimate $\theta^{(0)}$, we iteratively update the ability estimate using:
$$\theta^{(t+1)} = \theta^{(t)} - \frac{TCC(\theta^{(t)}) - X_{\text{adjusted}}}{TCC'(\theta^{(t)})}$$

where the starting value is based on the logit of the proportion correct adjusted by the average item difficulty:
$$\theta^{(0)} = \ln \left( \frac{p}{1-p} \right) + \bar{d}, \quad p = \frac{X_{\text{adjusted}}}{M}, \quad \bar{d} = \frac{1}{M} \sum_{i=1}^N \sum_{j=1}^{m_i} \delta_{ij}$$

Convergence is rapid and typically completes in under 5 iterations.

In [ ]:
import numpy as np
import pandas as pd
import rasch_mixed_item_cut as rmic

# Define a mixed test consisting of 10 items:
# - 6 dichotomous items (represented by difficulty 'b' or 'difficulty')
# - 4 polytomous PCM items (represented by step difficulties 'steps')
items = [
    {"difficulty": -1.2, "label": "Dichotomous Item 1"},
    {"difficulty": -0.5, "label": "Dichotomous Item 2"},
    {"difficulty": 0.0, "label": "Dichotomous Item 3"},
    {"difficulty": 0.5, "label": "Dichotomous Item 4"},
    {"difficulty": 1.0, "label": "Dichotomous Item 5"},
    {"difficulty": 1.8, "label": "Dichotomous Item 6"},
    
    # Polytomous PCM items
    {"steps": [-1.0, 0.5], "label": "PCM Item 7 (3 categories: 0, 1, 2)"},
    {"steps": [-0.5, 0.2, 1.2], "label": "PCM Item 8 (4 categories: 0, 1, 2, 3)"},
    {"difficulty": 0.2, "steps": [-0.8, 0.8], "label": "PCM Item 9 (RSM-style thresholds relative to 0.2)"},
    {"steps": [0.0, 1.0, 2.0], "label": "PCM Item 10 (4 categories: 0, 1, 2, 3)"}
]

# Parse item parameters into standardized PCM absolute step difficulties
parsed_items = rmic.parse_items(items)
max_score = sum(len(steps) for steps in parsed_items)

print(f"Test Structure Summary:")
print(f"- Total Items: {len(parsed_items)}")
print(f"- Max Raw Score: {max_score}")
print("\nParsed Step Difficulties (absolute δ_ij for each item):")
for idx, steps in enumerate(parsed_items):
    label = items[idx].get("label", f"Item {idx+1}")
    print(f"  Item {idx+1:2d} ({label}): {steps}")

### Raw-Score-to-Logit Conversion Table

The table below shows the complete conversion mapping for every possible raw score on this test form. This represents the theoretical mapping equivalent to Winsteps' `SCOREFILE` output.

In [ ]:
# Generate the complete conversion table
conversion_df = rmic.generate_conversion_table(items, extrscore=0.3)

# Format and display beautifully
df_styled = conversion_df.copy()
df_styled.rename(columns={
    "Raw Score": "Raw Score (X)",
    "Adjusted Score": "Adjusted Score (X_adj)",
    "Logit Measure": "Ability Estimate (θ)",
    "Model SE": "Standard Error (SE)",
    "Converged": "NR Converged"
}, inplace=True)

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
display(df_styled)

## Iterative Solver Demonstration (Newton-Raphson Trace)

Let's trace the Newton-Raphson iteration process step-by-step for a specific raw cut score of `8` to visualize the convergence trajectory.

In [ ]:
raw_cut = 8
extrscore = 0.3
tol = 1e-7
max_iter = 100

# 1. Apply adjustment
if raw_cut <= 0.0:
    adjusted_score = extrscore
elif raw_cut >= max_score:
    adjusted_score = max_score - extrscore
else:
    adjusted_score = raw_cut

# 2. Calculate initial theta
p = adjusted_score / max_score
all_steps = [s for steps in parsed_items for s in steps]
mean_difficulty = np.mean(all_steps)
theta = np.log(p / (1.0 - p)) + mean_difficulty

print(f"--- Newton-Raphson Trace for Raw Cut = {raw_cut} ---")
print(f"Max possible test score M:  {max_score}")
print(f"Adjusted score target:      {adjusted_score:.4f}")
print(f"Average step difficulty:     {mean_difficulty:.4f} logits")
print(f"Initial theta estimate θ^0:  {theta:.4f} logits\n")
print(f"{'Iter':<6} | {'Ability (θ)':<12} | {'Expected Score':<15} | {'Information (I)':<15} | {'Step size':<12}")
print("-" * 70)

converged = False
for i in range(max_iter):
    tcc, slope = rmic.compute_tcc_and_slope(theta, parsed_items)
    diff = tcc - adjusted_score
    step = diff / slope
    theta_new = theta - step
    
    print(f"{i+1:<6d} | {theta:<12.6f} | {tcc:<15.6f} | {slope:<15.6f} | {step:<12.6f}")
    
    if abs(step) < tol:
        theta = theta_new
        converged = True
        break
    theta = theta_new

se = 1.0 / np.sqrt(slope)
print("-" * 70)
print(f"Final Solution:")
print(f"- Converged Logit Cut: {theta:.6f} logits")
print(f"- Model Standard Error: {se:.6f} logits")
print(f"- Total Iterations:     {i+1}")

# Verify with module function
module_res = rmic.raw_to_logit(raw_cut, parsed_items, extrscore=extrscore)
assert abs(module_res["logit"] - theta) < 1e-9, "Trace calculation does not match module output!"

## Visualization of Cut Score on TCC & TIF Curves

Below, we visualize the position of the raw cut score $X = 8$ and its mapped logit cut score $\theta \approx 0.3402$ on both the Test Characteristic Curve (TCC) and the Test Information Function (TIF).

In [ ]:
# Plot the TCC and TIF curves inline
# Visualizing the cut score of interest using cut_score=raw_cut
%matplotlib inline
fig = rmic.plot_tcc_and_tif(items, conversion_df=conversion_df, cut_score=raw_cut)